# 🚀 Crypto News Classifier - Training Notebook

Fine-tune BERT untuk klasifikasi berita crypto menjadi 3 level:
- **PENTING**: Berita yang sangat berpengaruh (ETF, hack, regulasi)
- **LUMAYAN**: Berita yang cukup berpengaruh
- **BIASA**: Berita rutin/harian

---

## 📋 Daftar Isi

1. [Setup & Install](#1-setup--install)
2. [Upload Data](#2-upload-data)
3. [Explorasi Data](#3-eksplorasi-data)
4. [Preprocessing](#4-preprocessing)
5. [Inisialisasi Model](#5-inisialisasi-model)
6. [Training](#6-training)
7. [Evaluasi](#7-evaluasi)
8. [Inference Test](#8-inference-test)
9. [Download Model](#9-download-model)

---

<a id='1-setup--install'></a>
## 1. Setup & Install

Install dependencies yang dibutuhkan.

In [ ]:
#@title 1.1 Install Packages { display-mode: "form" }
#@markdown Jalankan cell ini untuk install packages
!pip install -q torch transformers scikit-learn accelerate

In [ ]:
#@title 1.2 Check GPU { display-mode: "form" }
#@markdown Pastikan GPU tersedia (Runtime → Change runtime type → T4 GPU)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ GPU tidak tersedia! Jalankan: Runtime → Change runtime type → T4 GPU")

<a id='2-upload-data'></a>
## 2. Upload Data

Upload file `labeled_articles.jsonl` dari lokal ke Colab.

In [ ]:
#@title 2.1 Upload labeled_articles.jsonl { display-mode: "form" }
#@markdown Upload file labeled_articles.jsonl dari folder:
#@markdown `tradingagents/news_classifier/data/cache/labeled_articles.jsonl`
from google.colab import files
import os

print("📤 Upload file labeled_articles.jsonl:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"✅ File uploaded: {filename} ({len(uploaded[filename])} bytes)")
    # Rename if needed
    if filename != "labeled_articles.jsonl":
        os.rename(filename, "labeled_articles.jsonl")
        print(f"   Renamed to: labeled_articles.jsonl")

print("\n📁 Files in current directory:")
!ls -la *.jsonl

<a id='3-eksplorasi-data'></a>
## 3. Explorasi Data

Analisis distribusi label dan contoh data.

In [ ]:
#@title 3.1 Load & Analyze Data { display-mode: "form" }
import json
from collections import Counter
import matplotlib.pyplot as plt

# Label mapping
LABEL_MAP = {"BIASA": 0, "LUMAYAN": 1, "PENTING": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

# Load data
articles = []
with open("labeled_articles.jsonl", "r") as f:
    for line in f:
        article = json.loads(line.strip())
        if article.get("label") in LABEL_MAP:
            articles.append(article)

print(f"📊 Total articles: {len(articles)}")

# Distribution
labels = [a["label"] for a in articles]
dist = Counter(labels)

print("\n📈 Label Distribution:")
for label in ["BIASA", "LUMAYAN", "PENTING"]:
    count = dist[label]
    pct = 100 * count / len(articles)
    bar = "█" * int(pct / 2)
    print(f"  {label:10s}: {count:4d} ({pct:5.1f}%) {bar}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.bar(["BIASA", "LUMAYAN", "PENTING"], [dist["BIASA"], dist["LUMAYAN"], dist["PENTING"]], color=colors)
ax.set_ylabel("Count")
ax.set_title("Label Distribution")
for bar, count in zip(bars, [dist["BIASA"], dist["LUMAYAN"], dist["PENTING"]]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(count), ha="center")
plt.tight_layout()
plt.show()

In [ ]:
#@title 3.2 Contoh Data per Label { display-mode: "form" }
#@markdown Tampilkan 2 contoh artikel per label
for label in ["PENTING", "LUMAYAN", "BIASA"]:
    print(f"\n{'='*60}")
    print(f"📰 {label} (contoh)")
    print(f"{'='*60}")
    examples = [a for a in articles if a["label"] == label][:2]
    for i, ex in enumerate(examples, 1):
        print(f"\n  [{i}] {ex.get('title', 'N/A')[:80]}")
        print(f"      Source: {ex.get('source', 'N/A')[:50]}")

<a id='4-preprocessing'></a>
## 4. Preprocessing

Split data dan buat PyTorch Dataset.

In [ ]:
#@title 4.1 Split Data { display-mode: "form" }
from sklearn.model_selection import train_test_split

def preprocess(article):
    title = article.get("title", "")
    desc = article.get("description", "")
    text = f"TITLE: {title} CONTENT: {desc}"
    return text[:512]  # Truncate to 512 tokens

texts = [preprocess(a) for a in articles]
labels = [LABEL_MAP[a["label"]] for a in articles]

# Split: 80% train, 10% val, 10% test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"📊 Data Split:")
print(f"  Train: {len(train_texts)} articles")
print(f"  Val:   {len(val_texts)} articles")
print(f"  Test:  {len(test_texts)} articles")

# Verify distribution per split
for name, lbls in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    dist = Counter(lbls)
    print(f"\n  {name}:", {ID_TO_LABEL[k]: v for k, v in dist.items()})

In [ ]:
#@title 4.2 Create PyTorch Dataset { display-mode: "form" }
from torch.utils.data import Dataset, DataLoader

class CryptoNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("✅ CryptoNewsDataset class defined")

<a id='5-inisialisasi-model'></a>
## 5. Inisialisasi Model

Load pre-trained BERT dan tambah classification head.

In [ ]:
#@title 5.1 Load BERT Model { display-mode: "form" }
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bert-base-uncased"  #@param ["bert-base-uncased", "distilbert-base-uncased", "albert-base-v2"]
MAX_LENGTH = 256  # @param {type:"integer"}
BATCH_SIZE = 8    # @param {type:"integer"}

print(f"📥 Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    problem_type="single_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ Model loaded on {device}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Create DataLoaders
train_dataset = CryptoNewsDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = CryptoNewsDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = CryptoNewsDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"\n📦 DataLoaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

<a id='6-training'></a>
## 6. Training

Fine-tune BERT dengan class weighting untuk mengatasi imbalance.

In [ ]:
#@title 6.1 Training Configuration { display-mode: "form" }
from transformers import get_linear_schedule_with_warmup
import numpy as np

#@markdown ### Hyperparameters
EPOCHS = 3        # @param {type:"integer"}
LEARNING_RATE = 2e-5  # @param {type:"number"}
WARMUP_RATIO = 0.1    # @param {type:"number"}
WEIGHT_DECAY = 0.01   # @param {type:"number"}

# Calculate class weights for imbalance
class_counts = [Counter(train_labels).get(i, 1) for i in range(3)]
class_weights = torch.tensor([sum(class_counts) / (3 * c) for c in class_counts]).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps
)

print("⚙️ Training Configuration:")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Total Steps: {total_steps}")
print(f"   Class Weights: {[f'{w:.2f}' for w in class_weights.tolist()]}")
print(f"   Device: {device}")

In [ ]:
#@title 6.2 Run Training { display-mode: "form" }
from sklearn.metrics import f1_score
import time

best_val_f1 = 0
history = []

print("🚀 Starting training...")
print("=" * 70)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # ========== TRAIN ==========
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    # ========== VALIDATE ==========
    model.eval()
    all_preds = []
    all_labels = []
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    val_loss = val_loss / len(val_loader)
    val_acc = sum(1 for p, l in zip(all_preds, all_labels) if p == l) / len(all_labels)
    val_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_time = time.time() - epoch_start

    history.append({"epoch": epoch+1, "train_loss": train_loss, "val_loss": val_loss, "val_f1": val_f1})

    print(f"Epoch {epoch+1}/{EPOCHS} ({epoch_time:.1f}s)")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✅ New best model saved (F1: {val_f1:.4f})")

    print("-" * 70)

print(f"\n🏆 Training complete! Best Val F1: {best_val_f1:.4f}")

In [ ]:
#@title 6.3 Plot Training History { display-mode: "form" }
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = [h["epoch"] for h in history]
ax1.plot(epochs, [h["train_loss"] for h in history], "o-", label="Train")
ax1.plot(epochs, [h["val_loss"] for h in history], "o-", label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss over Epochs")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, [h["val_f1"] for h in history], "o-", color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1 Macro")
ax2.set_title("Validation F1 over Epochs")
ax2.grid(True)

plt.tight_layout()
plt.show()

<a id='7-evaluasi'></a>
## 7. Evaluasi

Evaluasi model terbaik pada test set.

In [ ]:
#@title 7.1 Evaluate on Test Set { display-mode: "form" }
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print("=" * 50)
print("📊 TEST RESULTS")
print("=" * 50)
print(classification_report(all_labels, all_preds, target_names=["BIASA", "LUMAYAN", "PENTING"]))

In [ ]:
#@title 7.2 Confusion Matrix { display-mode: "form" }
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["BIASA", "LUMAYAN", "PENTING"],
            yticklabels=["BIASA", "LUMAYAN", "PENTING"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

<a id='8-inference-test'></a>
## 8. Inference Test

Test model dengan input manual.

In [ ]:
#@title 8.1 Test Inference { display-mode: "form" }
#@markdown Ketik berita crypto untuk di-klasifikasi
news_title = "Bitcoin ETF approved by SEC in historic decision" # @param {type:"string"}
news_content = "The SEC has approved the first Bitcoin ETF, marking a historic milestone for cryptocurrency markets." # @param {type:"string"}

def classify_news(title, content):
    text = f"TITLE: {title} CONTENT: {content}"
    encoding = tokenizer(
        text[:512],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_idx = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_idx].item()

    label = ID_TO_LABEL[pred_idx]
    probs_dict = {ID_TO_LABEL[i]: round(probs[0][i].item(), 4) for i in range(3)}

    return label, confidence, probs_dict

# Classify
label, confidence, probs = classify_news(news_title, news_content)

print("📰 Input:")
print(f"   Title: {news_title}")
print(f"   Content: {news_content[:100]}...")
print(f"\n🎯 Prediction:")
print(f"   Label: {label}")
print(f"   Confidence: {confidence:.2%}")
print(f"   Probabilities:")
for l, p in sorted(probs.items(), key=lambda x: -x[1]):
    bar = "█" * int(p * 50)
    print(f"     {l:10s}: {p:.2%} {bar}")

In [ ]:
#@title 8.2 Batch Test { display-mode: "form" }
#@markdown Test beberapa berita sekaligus
test_cases = [
    ("Major exchange hacked, $500M stolen", "Binance has been hacked for $500 million in Bitcoin"),
    ("Bitcoin price up 2% today", "Bitcoin gained 2% in the last 24 hours"),
    ("New partnership announced", "Project X partners with Y for DeFi integration"),
    ("SEC sues major crypto company", "SEC files lawsuit against major crypto exchange for securities violations"),
    ("Weekly market update", "Crypto markets showed mixed performance this week"),
]

print(f"{'='*80}")
print(f"{'Title':<50} {'Label':<10} {'Conf':<8}")
print(f"{'='*80}")

for title, content in test_cases:
    label, confidence, _ = classify_news(title, content)
    print(f"{title[:48]:<50} {label:<10} {confidence:<8.2%}")

print(f"{'='*80}")

<a id='9-download-model'></a>
## 9. Download Model

Download model yang sudah di-training ke lokal.

In [ ]:
#@title 9.1 Download best_model.pt { display-mode: "form" }
#@markdown Download model ke lokal, lalu copy ke:
#@markdown `tradingagents/news_classifier/pretrained/best_model.pt`
from google.colab import files

print("📥 Downloading best_model.pt...")
files.download("best_model.pt")
print("\n✅ Model downloaded!")
print("\n📋 Next steps:")
print("   1. Copy best_model.pt ke tradingagents/news_classifier/pretrained/")
print("   2. Jalankan: python -m tradingagents.news_classifier.webhook.server")
print("   3. Test: curl -X POST http://localhost:8000/classify -d '{\"title\":\"...\"}'")

In [ ]:
#@title 9.2 Save to Google Drive (Optional) { display-mode: "form" }
#@markdown Simpan model ke Google Drive untuk backup
# from google.colab import drive
# drive.mount('/content/drive')
# !cp best_model.pt /content/drive/MyDrive/best_model.pt
# print("✅ Model saved to Google Drive")

---

## 📝 Summary

| Step | Status |
|------|--------|
| 1. Setup & Install | ✅ |
| 2. Upload Data | ✅ |
| 3. Explorasi Data | ✅ |
| 4. Preprocessing | ✅ |
| 5. Inisialisasi Model | ✅ |
| 6. Training | ✅ |
| 7. Evaluasi | ✅ |
| 8. Inference Test | ✅ |
| 9. Download Model | ✅ |

### Hasil Training

Setelah training, copy `best_model.pt` ke:
```
tradingagents/news_classifier/pretrained/best_model.pt
```

### Cara Pakai

```bash
# Start webhook server
python -m tradingagents.news_classifier.webhook.server

# Test classification
curl -X POST http://localhost:8000/classify \
  -H "Content-Type: application/json" \
  -d '{"title": "Bitcoin ETF approved by SEC"}'
```

---

Built with ❤️ for TradingAgents Stellar Hackathon